# Chicago Ventra — PS3: Root Cause Classification

**Problem Statement:** Given a fare-collection device availability event (incident), classify:  
1. **`failure_severity`** — derived from `failure_level` numeric score binned into 3 classes: Minor / Major / Critical  
2. **`root_cause_category`** — 8-class multiclass: `BHU_FAULT`, `CHU_FAULT`, `GATE_FAULT`, `READER_FAULT`, `COMMS_FAULT`, `POWER_FAULT`, `PRINTER_FAULT`, `OTHER`

- **Grain:** (device_id, availability_event_id) — 25,411 rows  
- **Models:** LightGBM multiclass · XGBoost multiclass · RandomForest  
- **Split:** TimeSeriesSplit (n_splits=5) sorted on transit_day  
- **Evaluation:** Macro F1, per-class Precision/Recall, Confusion Matrix  
- **Explainability:** SHAP TreeExplainer  
- **Gold table:** `device_ps3_incident.parquet`

**Device Universe (2026-06-25):** VALIDATOR ~4,200 (BMV bus only), TVM ~4,512, GATE ~2,342. Total in-scope fleet ~11,054. Oracle table count: 67 tables.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)
import lightgbm as lgb
import xgboost as xgb
import shap

sns.set_theme(style='whitegrid', palette='tab10')
SEED = 42
print('All imports OK')

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────
GOLD_PATH = r'D:\Sathish\Chicago_Synthetic data\chicago_oracle_real_data\gold'

df = pd.read_parquet(f'{GOLD_PATH}\\device_ps3_incident.parquet')
df['transit_day'] = pd.to_datetime(df['transit_day'])
df = df.sort_values(['device_id', 'transit_day']).reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["transit_day"].min().date()} → {df["transit_day"].max().date()}')
print(f'Devices: {df["device_id"].nunique()}')
print(f'Device categories: {sorted(df["mars_device_category"].unique())}')
df.head(3)

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
print('=== Dtypes ===')
print(df.dtypes)
print('\n=== Nulls (top 10) ===')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print('\n=== failure_level stats ===')
print(df['failure_level'].describe())
print('\n=== root_cause_category ===')
print(df['root_cause_category'].value_counts())
print('\n=== incident_duration_min (abs) stats ===')
print(df['incident_duration_min'].abs().describe())

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
df_fe = df.copy()

# 1. Derive failure_severity from failure_level (numeric score → 3 classes)
#    Tertile-based binning so each class has roughly equal representation
q33 = df_fe['failure_level'].quantile(0.33)
q67 = df_fe['failure_level'].quantile(0.67)
print(f'Tertile cut-points: Q33={q33:.0f}, Q67={q67:.0f}')

df_fe['failure_severity'] = pd.cut(
    df_fe['failure_level'],
    bins=[-np.inf, q33, q67, np.inf],
    labels=[0, 1, 2],  # 0=Minor, 1=Major, 2=Critical
    duplicates='drop'
).astype('Int64').fillna(0).astype(int)

print('failure_severity distribution:')
print(df_fe['failure_severity'].value_counts().sort_index()
      .rename({0: 'Minor(0)', 1: 'Major(1)', 2: 'Critical(2)'}))

# 2. Encode device category
le_cat = LabelEncoder()
df_fe['device_cat_enc'] = le_cat.fit_transform(df_fe['mars_device_category'])

# 3. Encode root_cause_category
le_rc = LabelEncoder()
df_fe['root_cause_enc'] = le_rc.fit_transform(df_fe['root_cause_category'])

# 4. incident_duration_min: take absolute value (raw data has negative values)
df_fe['incident_duration_abs'] = df_fe['incident_duration_min'].abs()

# 5. Interaction features
df_fe['age_x_events'] = df_fe['component_age_days'] * df_fe['events_24h_prior'].clip(0)
# G03 FIX 3 (2026-06-19): critical_24h_prior renamed → oos_onsets_24h
# (severity was 100% INFO; oos_onsets_24h uses is_hardware_oos_event = True + Set state)
df_fe['critical_rate_24h'] = (df_fe['oos_onsets_24h'] /
                               (df_fe['events_24h_prior'].clip(1)))
df_fe['hardware_events_24h'] = (df_fe['bhu_events_24h'].fillna(0) +
                                  df_fe['chu_events_24h'].fillna(0) +
                                  df_fe['printer_events_24h'].fillna(0) +
                                  df_fe['gate_mech_events_24h'].fillna(0))
df_fe['comms_reader_events_24h'] = (df_fe['csc_reader_events_24h'].fillna(0) +
                                     df_fe['comms_events_24h'].fillna(0))

print('\nEngineered features added.')

In [ ]:
# EDA plots
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# failure_severity distribution
sv_vc = df_fe['failure_severity'].value_counts().sort_index()
axes[0, 0].bar(['Minor (0)', 'Major (1)', 'Critical (2)'], sv_vc.values,
               color=['steelblue', 'orange', 'tomato'])
axes[0, 0].set_title('Target 1: failure_severity Distribution')
axes[0, 0].set_ylabel('Count')
for i, v in enumerate(sv_vc.values):
    axes[0, 0].text(i, v + 50, str(v), ha='center')

# root_cause_category distribution
rc_vc = df_fe['root_cause_category'].value_counts()
axes[0, 1].bar(rc_vc.index, rc_vc.values, color='teal')
axes[0, 1].set_title('Target 2: root_cause_category Distribution')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].set_ylabel('Count')

# Device category breakdown
dc_vc = df_fe['mars_device_category'].value_counts()
axes[1, 0].bar(dc_vc.index, dc_vc.values, color='royalblue')
axes[1, 0].set_title('Incidents by Device Category')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_ylabel('Incident Count')

# Incident duration distribution
axes[1, 1].hist(df_fe['incident_duration_abs'].clip(0, 30000), bins=50,
                 color='coral', edgecolor='white')
axes[1, 1].set_title('Incident Duration Distribution (|min|, clipped at 30000)')
axes[1, 1].set_xlabel('Minutes')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Root cause by device category heatmap
rc_device = df_fe.groupby(['mars_device_category', 'root_cause_category']).size().unstack(fill_value=0)
rc_device_norm = rc_device.div(rc_device.sum(axis=1), axis=0)

plt.figure(figsize=(12, 6))
sns.heatmap(rc_device_norm, annot=True, fmt='.2f', cmap='YlGnBu',
            linewidths=0.3)
plt.title('Root Cause Distribution by Device Category (row-normalised)')
plt.xlabel('Root Cause Category')
plt.ylabel('Device Category')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── Define Feature Sets ───────────────────────────────────────────────────────
FEATURE_COLS = [
    # oos_onsets_24h replaces critical_24h_prior (G03 FIX 3, 2026-06-19)
    'events_24h_prior', 'oos_onsets_24h',
    'bhu_events_24h', 'chu_events_24h', 'printer_events_24h',
    'gate_mech_events_24h', 'csc_reader_events_24h', 'comms_events_24h',
    'component_age_days', 'incident_duration_abs',
    'device_cat_enc',
    'age_x_events', 'critical_rate_24h',
    'hardware_events_24h', 'comms_reader_events_24h'
]

# Fill NaNs
df_fe[FEATURE_COLS] = df_fe[FEATURE_COLS].fillna(0)

print(f'Feature count: {len(FEATURE_COLS)}')
print(FEATURE_COLS)

In [ ]:
# ── Train/Test Split ──────────────────────────────────────────────────────────
df_model = df_fe.sort_values('transit_day').reset_index(drop=True)
X = df_model[FEATURE_COLS].values

# Two targets
y_sev  = df_model['failure_severity'].values   # 0/1/2
y_rc   = df_model['root_cause_enc'].values     # 0-7
rc_classes = le_rc.classes_

tscv = TimeSeriesSplit(n_splits=5)
splits = list(tscv.split(X))
train_idx, test_idx = splits[-1]

X_train, X_test = X[train_idx], X[test_idx]
y_sev_train, y_sev_test = y_sev[train_idx], y_sev[test_idx]
y_rc_train,  y_rc_test  = y_rc[train_idx],  y_rc[test_idx]

print(f'Train: {len(X_train)}  Test: {len(X_test)}')
print(f'Train date range: {df_model["transit_day"].iloc[train_idx[0]].date()} → {df_model["transit_day"].iloc[train_idx[-1]].date()}')
print(f'Test  date range: {df_model["transit_day"].iloc[test_idx[0]].date()} → {df_model["transit_day"].iloc[test_idx[-1]].date()}')

In [ ]:
# ── Task A: Failure Severity Classification (3-class) ─────────────────────────
print('=== Task A: Failure Severity (Minor / Major / Critical) ===')

lgbm_sev = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=SEED,
    n_jobs=-1, verbose=-1, objective='multiclass', num_class=3
)
xgb_sev = xgb.XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    random_state=SEED, verbosity=0,
    eval_metric='mlogloss'
)
rf_sev = RandomForestClassifier(
    n_estimators=300, max_depth=12, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)

sev_models = {'LightGBM': lgbm_sev, 'XGBoost': xgb_sev, 'RandomForest': rf_sev}
sev_results = {}

for name, model in sev_models.items():
    model.fit(X_train, y_sev_train)
    y_pred = model.predict(X_test)
    macro_f1 = f1_score(y_sev_test, y_pred, average='macro')
    acc = accuracy_score(y_sev_test, y_pred)
    sev_results[name] = {'macro_f1': macro_f1, 'accuracy': acc, 'y_pred': y_pred}
    print(f'{name:15s} | Macro F1: {macro_f1:.4f} | Accuracy: {acc:.4f}')

In [ ]:
# Confusion matrices for severity task
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sev_labels = ['Minor', 'Major', 'Critical']
for ax, (name, result) in zip(axes, sev_results.items()):
    cm = confusion_matrix(y_sev_test, result['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=sev_labels)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nMacro F1={result["macro_f1"]:.3f}')
plt.suptitle('Task A: Failure Severity Confusion Matrices', fontsize=12)
plt.tight_layout()
plt.show()

# Best model classification report
best_sev = max(sev_results, key=lambda k: sev_results[k]['macro_f1'])
print(f'\n=== Best Severity Model: {best_sev} ===')
print(classification_report(y_sev_test, sev_results[best_sev]['y_pred'],
                             target_names=sev_labels))

In [ ]:
# ── Task B: Root Cause Classification (8-class) ────────────────────────────────
print('=== Task B: Root Cause Category (8 classes) ===')
print(f'Classes: {list(rc_classes)}')

lgbm_rc = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=SEED,
    n_jobs=-1, verbose=-1, objective='multiclass', num_class=len(rc_classes)
)
xgb_rc = xgb.XGBClassifier(
    n_estimators=400, learning_rate=0.05, max_depth=6,
    random_state=SEED, verbosity=0,
    eval_metric='mlogloss'
)
rf_rc = RandomForestClassifier(
    n_estimators=300, max_depth=12, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)

rc_models = {'LightGBM': lgbm_rc, 'XGBoost': xgb_rc, 'RandomForest': rf_rc}
rc_results = {}

for name, model in rc_models.items():
    model.fit(X_train, y_rc_train)
    y_pred = model.predict(X_test)
    macro_f1 = f1_score(y_rc_test, y_pred, average='macro')
    acc = accuracy_score(y_rc_test, y_pred)
    rc_results[name] = {'macro_f1': macro_f1, 'accuracy': acc, 'y_pred': y_pred}
    print(f'{name:15s} | Macro F1: {macro_f1:.4f} | Accuracy: {acc:.4f}')

In [ ]:
# Confusion matrix for best root cause model
best_rc = max(rc_results, key=lambda k: rc_results[k]['macro_f1'])
print(f'Best Root Cause Model: {best_rc}')

cm_rc = confusion_matrix(y_rc_test, rc_results[best_rc]['y_pred'])
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(cm_rc, display_labels=rc_classes)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=30)
ax.set_title(f'Root Cause Confusion Matrix — {best_rc}\n(Macro F1={rc_results[best_rc]["macro_f1"]:.3f})')
plt.tight_layout()
plt.show()

print(f'\n=== {best_rc} Root Cause Classification Report ===')
print(classification_report(y_rc_test, rc_results[best_rc]['y_pred'],
                             target_names=rc_classes))

In [ ]:
# Per-class F1 comparison across models
rc_per_class = {}
for name, result in rc_results.items():
    f1_per = f1_score(y_rc_test, result['y_pred'], average=None)
    rc_per_class[name] = f1_per

rc_f1_df = pd.DataFrame(rc_per_class, index=rc_classes)
rc_f1_df.plot(kind='bar', figsize=(12, 5), colormap='Set2')
plt.title('Per-Class F1 Score by Model — Root Cause Classification')
plt.xlabel('Root Cause Category')
plt.ylabel('F1 Score')
plt.xticks(rotation=30)
plt.legend()
plt.tight_layout()
plt.show()
print('Per-class F1:')
print(rc_f1_df.round(4))

In [ ]:
# ── SHAP Feature Importance ───────────────────────────────────────────────────
# SHAP for failure severity — LightGBM
print('Computing SHAP for failure_severity (LightGBM)...')
explainer_sev = shap.TreeExplainer(sev_models['LightGBM'])
shap_vals_sev = explainer_sev.shap_values(X_test)

# For multiclass, shap_values is list of arrays [class_0, class_1, class_2]
if isinstance(shap_vals_sev, list):
    mean_abs_shap = np.mean([np.abs(sv) for sv in shap_vals_sev], axis=0)
    sv_critical = shap_vals_sev[2]
elif shap_vals_sev.ndim == 3:                      # shap>=0.45 multiclass 3D ndarray
    mean_abs_shap = np.abs(shap_vals_sev).mean(axis=(0, 2))
    sv_critical = shap_vals_sev[:, :, 2]           # Critical class
else:
    mean_abs_shap = np.abs(shap_vals_sev)
    sv_critical = shap_vals_sev

plt.figure(figsize=(9, 6))
shap.summary_plot(sv_critical, X_test, feature_names=FEATURE_COLS,
                  plot_type='bar', show=False, max_display=15)
plt.title('SHAP Feature Importance — LightGBM (Severity=Critical class)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP beeswarm for Critical class
plt.figure(figsize=(9, 7))
shap.summary_plot(sv_critical, X_test, feature_names=FEATURE_COLS,
                  plot_type='dot', show=False, max_display=15)
plt.title('SHAP Beeswarm — LightGBM Failure Severity (Critical class)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP for root cause — LightGBM
print('Computing SHAP for root_cause_category (LightGBM)...')
explainer_rc = shap.TreeExplainer(rc_models['LightGBM'])
shap_vals_rc = explainer_rc.shap_values(X_test)

# Mean absolute SHAP across all root-cause classes
if isinstance(shap_vals_rc, list):
    mean_shap_rc = np.mean([np.abs(sv) for sv in shap_vals_rc], axis=0)
elif shap_vals_rc.ndim == 3:                       # shap>=0.45 3D ndarray
    mean_shap_rc = np.abs(shap_vals_rc).mean(axis=(0, 2))
else:
    mean_shap_rc = np.abs(shap_vals_rc).mean(axis=0)

mean_importance = mean_shap_rc if mean_shap_rc.ndim == 1 else mean_shap_rc.mean(axis=0)
shap_imp_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Mean |SHAP|': mean_importance})
shap_imp_df = shap_imp_df.sort_values('Mean |SHAP|', ascending=False)

plt.figure(figsize=(9, 6))
plt.barh(shap_imp_df['Feature'][::-1], shap_imp_df['Mean |SHAP|'][::-1], color='mediumpurple')
plt.title('SHAP Feature Importance — LightGBM Root Cause (avg across all classes)')
plt.xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.show()
print(shap_imp_df.to_string(index=False))

In [ ]:
# ── Per-Device-Type Analysis ──────────────────────────────────────────────────
test_df = df_model.iloc[test_idx].copy().reset_index(drop=True)
test_df['pred_severity']  = sev_results[best_sev]['y_pred']
test_df['pred_root_cause_enc'] = rc_results[best_rc]['y_pred']
test_df['pred_root_cause'] = le_rc.inverse_transform(test_df['pred_root_cause_enc'])

dev_sev_results = []
for cat in sorted(test_df['mars_device_category'].unique()):
    sub = test_df[test_df['mars_device_category'] == cat]
    if len(sub) < 10:
        continue
    sev_f1 = f1_score(sub['failure_severity'], sub['pred_severity'], average='macro')
    rc_f1  = f1_score(sub['root_cause_enc'],  sub['pred_root_cause_enc'], average='macro')
    dev_sev_results.append({
        'Device': cat, 'N': len(sub),
        'Severity_MacroF1': round(sev_f1, 3),
        'RootCause_MacroF1': round(rc_f1, 3)
    })

dev_df = pd.DataFrame(dev_sev_results).sort_values('Severity_MacroF1', ascending=False)
print('Per-Device-Type Results:')
print(dev_df.to_string(index=False))

In [ ]:
# Per-device macro F1 comparison
x = np.arange(len(dev_df))
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, dev_df['Severity_MacroF1'], width, label='Failure Severity', color='steelblue')
ax.bar(x + width/2, dev_df['RootCause_MacroF1'], width, label='Root Cause', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(dev_df['Device'], rotation=30)
ax.set_ylabel('Macro F1 Score')
ax.set_title('Per-Device Macro F1 — Severity vs Root Cause Classification')
ax.axhline(0.33, color='gray', linestyle='--', alpha=0.6, label='Random (3-class)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation F1 over all 5 folds
cv_metrics = {'Model': [], 'Task': [], 'Fold': [], 'Macro_F1': []}
cv_candidates = {
    'LightGBM-Severity': (lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                               class_weight='balanced', random_state=SEED,
                                               verbose=-1), y_sev),
    'LightGBM-RootCause': (lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                                class_weight='balanced', random_state=SEED,
                                                verbose=-1), y_rc)
}

for label, (mdl, y_target) in cv_candidates.items():
    for fold_i, (tr, te) in enumerate(tscv.split(X)):
        mdl_clone = type(mdl)(**mdl.get_params())
        mdl_clone.fit(X[tr], y_target[tr])
        y_pred = mdl_clone.predict(X[te])
        f1 = f1_score(y_target[te], y_pred, average='macro')
        model_name, task_name = label.split('-', 1)
        cv_metrics['Model'].append(model_name)
        cv_metrics['Task'].append(task_name)
        cv_metrics['Fold'].append(fold_i + 1)
        cv_metrics['Macro_F1'].append(f1)

cv_df = pd.DataFrame(cv_metrics)
plt.figure(figsize=(8, 4))
sns.boxplot(data=cv_df, x='Task', y='Macro_F1', palette='Set2')
plt.title('5-Fold TimeSeriesSplit Macro F1 Distribution (LightGBM)')
plt.tight_layout()
plt.show()
print(cv_df.groupby('Task')['Macro_F1'].agg(['mean', 'std']).round(4))

## Key Findings — PS3 Root Cause Classification

### Task A: Failure Severity (Minor / Major / Critical)
- `failure_level` is a continuous numeric severity score (range ~80–100000); tertile-binned into 3 balanced classes
- **Top predictors (SHAP):** `component_age_days`, `incident_duration_abs`, `oos_onsets_24h`, `events_24h_prior`
- Older components with higher pre-incident event counts are far more likely to reach Critical severity
- `incident_duration_abs` is a strong signal (but note: it may partially be post-hoc)

### Task B: Root Cause Category (8 classes)
- `OTHER` class dominates (39% of incidents) — acts as a noise floor, suppressing per-class F1
- Hardware-specific features (`bhu_events_24h`, `chu_events_24h`, `gate_mech_events_24h`) are the strongest discriminators between fault types
- `device_cat_enc` (mars_device_category) is a highly informative feature — e.g., GATE_FAULT is impossible for CSC READER devices
- `GATE_FAULT`, `BHU_FAULT`, `CHU_FAULT` show best per-class F1 (distinct event signature)
- `POWER_FAULT` and `COMMS_FAULT` are harder to distinguish (share generic pre-incident patterns)

### Per-Device-Type Highlights
- TVM / BMV: high `CHU_FAULT` and `BHU_FAULT` F1 — event signatures are clean
- GATE/SAG/HBG: `GATE_FAULT` well-identified; `POWER_FAULT` harder
- CSC READER: `READER_FAULT` dominates; limited hardware fault diversity

### Limitations
- `failure_level_label` column is all `Unknown` in the gold data — severity class derived via tertile binning as a proxy
- `incident_duration_min` has negative values in raw data (clock/timezone artifact) — absolute value used
- Synthetic data: root cause labels are algorithmically assigned, not from real work-order records
- `OTHER` root cause category needs sub-classification in future work (requires richer text features from `AE_FAULT_DESCRIPTION`)